# Models Evaluation

## Traditional ML Models

In [51]:
from pathlib import Path

TRAIN_CSV = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/5000_Posts_Annotations - Combined_Dataset.csv")
TEST_CSV  = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/600_test_set.csv")

ID_COL    = "id"      
TITLE_COL = "title"
BODY_COL  = "body"


LABELS_COL = "Tags"

CANON_LABELS = [
    "Abuse", "Aggression", "Sexual", "Medical", "Mental Health",
    "Discrimination", "Pregnancy", "Not Applicable"
]

OUT_DIR = Path("/home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output dir:", OUT_DIR.resolve())

Output dir: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference


In [52]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Using device:", DEVICE)
print("✅ Output dir:", OUT_DIR.resolve())

✅ Using device: cuda
✅ Output dir: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference


In [53]:
import sys
print(sys.executable)

/home/ubuntu/TW_MultiLabel_SMP/tw_env/bin/python


In [54]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train:", train_df.shape, "Test:", test_df.shape)
display(train_df[[ID_COL, TITLE_COL, BODY_COL, LABELS_COL]].head(2))
display(test_df[[ID_COL, TITLE_COL, BODY_COL]].head(2))

Train: (4994, 6) Test: (600, 16)


,id,title,body,Tags
0,1ljxynj,Complications after abortion?,"Hi everyone, Ive read that abortions don’t cau...","Medical, Pregnancy, Mental Health"
1,1ljxtt8,Second MA abortion today and I'm absolutely te...,I'm having my second MA abortion today and I'm...,"Medical, Pregnancy, Mental Health"


,id,title,body
0,kg3jun,my assault ruins all of my relationships.,this is my first reddit post and i'm bad at ex...
1,77d66o,Me Too,After seeing all this hype over the #metoo thi...


In [55]:
def make_text(df):
    return (df[TITLE_COL].fillna("").astype(str) + " " + df[BODY_COL].fillna("").astype(str)).str.strip()

train_df["text"] = make_text(train_df)
test_df["text"]  = make_text(test_df)

print("text column created")

text column created


In [56]:
from sklearn.preprocessing import MultiLabelBinarizer

def parse_labels(x):
    return [t.strip() for t in str(x).split(",") if t.strip()]

train_df["labels_list"] = train_df[LABELS_COL].fillna("").apply(parse_labels)

mlb = MultiLabelBinarizer(classes=CANON_LABELS)
Y_train = mlb.fit_transform(train_df["labels_list"])

X_train = train_df["text"].astype(str).values
X_test  = test_df["text"].astype(str).values

print("Y_train:", Y_train.shape)
print("Labels:", list(mlb.classes_))

Y_train: (4994, 8)
Labels: ['Abuse', 'Aggression', 'Sexual', 'Medical', 'Mental Health', 'Discrimination', 'Pregnancy', 'Not Applicable']


In [57]:
from sklearn.model_selection import train_test_split

X_tr, X_val, Y_tr, Y_val = train_test_split(
    X_train, Y_train,
    test_size=0.2,
    random_state=42
)
print("Train split:", X_tr.shape, "Val split:", X_val.shape)

Train split: (3995,) Val split: (999,)


### TF-IDF + LR

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

lr_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True,
        strip_accents="unicode"
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            solver="saga",
            max_iter=3000,
            class_weight="balanced",
            n_jobs=-1
        )
    ))
])

lr_pipe.fit(X_tr, Y_tr)
print("Logistic Regression model trained")

/home/ubuntu/.local/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/ubuntu/.local/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Logistic Regression model trained


In [20]:
import numpy as np
from sklearn.metrics import f1_score

val_proba = lr_pipe.predict_proba(X_val)

thresholds = np.arange(0.20, 0.81, 0.05)
best_thr, best_micro = None, -1

for thr in thresholds:
    Yv_pred = (val_proba >= thr).astype(int)
    micro = f1_score(Y_val, Yv_pred, average="micro", zero_division=0)
    if micro > best_micro:
        best_micro = micro
        best_thr = thr

print("Best LR threshold:", best_thr, "Val micro-F1:", round(best_micro, 4))

Best LR threshold: 0.49999999999999994 Val micro-F1: 0.7757


In [21]:
import pandas as pd

test_proba_lr = lr_pipe.predict_proba(X_test)
Y_pred_lr = (test_proba_lr >= best_thr).astype(int)

pred_labels_lr = mlb.inverse_transform(Y_pred_lr)

pred_lr_df = pd.DataFrame({
    ID_COL: test_df[ID_COL].values,
    "pred_labels": [",".join(labels) if labels else "" for labels in pred_labels_lr],
    "threshold": best_thr
})

proba_cols = [f"proba_{lab}" for lab in mlb.classes_]
pred_lr_proba = pd.DataFrame(test_proba_lr, columns=proba_cols)

pred_lr_out = pd.concat([pred_lr_df, pred_lr_proba], axis=1)
LR_PATH = OUT_DIR / "predictions_tfidf_lr_test600.csv"
pred_lr_out.to_csv(LR_PATH, index=False)

print("Saved LR predictions:", LR_PATH.resolve())
display(pred_lr_out.head(3))

Saved LR predictions: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/predictions_tfidf_lr_test600.csv


,id,pred_labels,threshold,proba_Abuse,proba_Aggression,proba_Sexual,proba_Medical,proba_Mental Health,proba_Discrimination,proba_Pregnancy,proba_Not Applicable
0,kg3jun,"Abuse,Aggression,Sexual,Mental Health",0.5,0.862308,0.785555,0.742427,0.085497,0.749351,0.431987,0.105011,0.441694
1,77d66o,"Abuse,Aggression,Sexual,Mental Health",0.5,0.886917,0.736072,0.885444,0.163528,0.788677,0.350614,0.139335,0.254028
2,1lmsep9,Not Applicable,0.5,0.396362,0.465700,0.310570,0.233821,0.482805,0.265863,0.348081,0.579433


### TF-IDF + SVM

In [22]:
from sklearn.svm import LinearSVC

svm_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True,
        strip_accents="unicode"
    )),
    ("clf", OneVsRestClassifier(
        LinearSVC(class_weight="balanced")
    ))
])

svm_pipe.fit(X_train, Y_train)
print("Linear SVM trained")

Linear SVM trained


In [23]:
for i, label in enumerate(mlb.classes_):
    count = Y_train[:, i].sum()
    if count == 0:
        print(f"⚠️ Label '{label}' has ZERO training examples")

In [24]:
svm_scores = svm_pipe.decision_function(X_test)
Y_pred_svm = (svm_scores >= 0).astype(int)

pred_labels_svm = mlb.inverse_transform(Y_pred_svm)

pred_svm_df = pd.DataFrame({
    ID_COL: test_df[ID_COL].values,
    "pred_labels": [",".join(labels) if labels else "" for labels in pred_labels_svm],
    "threshold": 0.0
})

# Save raw decision scores per label too
score_cols = [f"score_{lab}" for lab in mlb.classes_]
pred_svm_scores = pd.DataFrame(svm_scores, columns=score_cols)

pred_svm_out = pd.concat([pred_svm_df, pred_svm_scores], axis=1)
SVM_PATH = OUT_DIR / "predictions_tfidf_svm_test600.csv"
pred_svm_out.to_csv(SVM_PATH, index=False)

print("Saved SVM predictions:", SVM_PATH.resolve())
display(pred_svm_out.head(3))

Saved SVM predictions: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/predictions_tfidf_svm_test600.csv


,id,pred_labels,threshold,score_Abuse,score_Aggression,score_Sexual,score_Medical,score_Mental Health,score_Discrimination,score_Pregnancy,score_Not Applicable
0,kg3jun,"Abuse,Sexual,Mental Health",0.0,0.741901,-0.836789,0.264626,-1.037100,0.641000,-0.375343,-1.122656,-0.335727
1,77d66o,"Abuse,Sexual,Mental Health",0.0,1.032490,-0.764224,0.925201,-0.607432,0.941310,-0.621776,-0.595512,-0.928600
2,1lmsep9,Not Applicable,0.0,-0.533150,-1.226374,-0.633979,-0.838246,-0.245532,-0.776147,-0.510205,0.325181


### Saving and Sanity Checks

In [25]:
import joblib, json

joblib.dump(lr_pipe, OUT_DIR / "tfidf_lr.joblib")
joblib.dump(svm_pipe, OUT_DIR / "tfidf_svm.joblib")
joblib.dump(mlb, OUT_DIR / "mlb.joblib")

meta = {
    "labels": list(mlb.classes_),
    "lr_best_threshold": float(best_thr),
    "tfidf": {"ngram_range": [1,2], "min_df": 2, "max_df": 0.9, "sublinear_tf": True}
}
with open(OUT_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved model artifacts to:", OUT_DIR.resolve())

Saved model artifacts to: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference


In [26]:
import numpy as np

lr_counts  = np.array([len(x) for x in pred_labels_lr])
svm_counts = np.array([len(x) for x in pred_labels_svm])

print("LR avg labels/post:", lr_counts.mean().round(2), "min:", lr_counts.min(), "max:", lr_counts.max())
print("SVM avg labels/post:", svm_counts.mean().round(2), "min:", svm_counts.min(), "max:", svm_counts.max())

print("\nLR label-count distribution:")
print(pd.Series(lr_counts).value_counts().sort_index())

print("\nSVM label-count distribution:")
print(pd.Series(svm_counts).value_counts().sort_index())

LR avg labels/post: 2.99 min: 0 max: 6
SVM avg labels/post: 2.42 min: 0 max: 6

LR label-count distribution:
0      4
1     38
2    131
3    238
4    165
5     23
6      1
Name: count, dtype: int64

SVM label-count distribution:
0      7
1     83
2    180
3    313
4     15
5      1
6      1
Name: count, dtype: int64


## Transformers

In [46]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import json

In [70]:
import numpy as np 

def build_ds(texts, labels=None):
    d = {"text": list(texts)}
    if labels is not None:
        d["labels"] = labels.astype(np.float32).tolist()
    return Dataset.from_dict(d)

def sigmoid(x):
    return 1/(1+np.exp(-x))

# def tune_threshold(val_logits, y_true):
#     probs = sigmoid(val_logits)
#     best_thr, best_micro = 0.5, -1
#     for thr in np.arange(0.20, 0.81, 0.05):
#         y_pred = (probs >= thr).astype(int)
#         micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
#         if micro > best_micro:
#             best_micro, best_thr = micro, thr
#     return float(best_thr), float(best_micro)

def tune_threshold(val_logits, y_true, grid=None):
    """
    Learn a separate threshold for each label using F1.
    
    val_logits: (N, L)
    y_true:     (N, L)
    """
    if grid is None:
        grid = np.arange(0.05, 0.95, 0.05)

    probs = sigmoid(val_logits)
    n_labels = probs.shape[1]

    best_thr = np.zeros(n_labels)
    best_f1  = np.zeros(n_labels)

    for j in range(n_labels):
        yj_true = y_true[:, j]

        # If label never appears in validation, keep default
        if yj_true.sum() == 0:
            best_thr[j] = 0.5
            best_f1[j] = 0.0
            continue

        scores = []
        for t in grid:
            yj_pred = (probs[:, j] >= t).astype(int)
            scores.append(
                f1_score(yj_true, yj_pred, zero_division=0)
            )

        idx = int(np.argmax(scores))
        best_thr[j] = grid[idx]
        best_f1[j]  = scores[idx]

    return best_thr, best_f1

In [65]:
def save_predictions(model_key, threshold, probs, pred_labels):
    """
    Saves:
      - id, model
      - threshold_global (if scalar)
      - thresholds_json (if vector, repeated per row)
      - thr_<label> columns (if vector, repeated per row)  ✅ nice for debugging
      - pred_labels
      - proba_<label> columns
    """
    model_dir = OUT_DIR / model_key
    model_dir.mkdir(parents=True, exist_ok=True)

    n = probs.shape[0]

    # --- handle scalar vs vector threshold safely ---
    is_scalar = np.isscalar(threshold)
    if is_scalar:
        threshold_global = float(threshold)
        thresholds_json = ""
        thr_cols_df = pd.DataFrame()  # empty
    else:
        thr_vec = np.asarray(threshold).astype(float).tolist()
        threshold_global = np.nan
        thresholds_json = json.dumps({lab: thr for lab, thr in zip(CANON_LABELS, thr_vec)})
        thr_cols_df = pd.DataFrame(
            {f"thr_{lab}": [thr]*n for lab, thr in zip(CANON_LABELS, thr_vec)}
        )

    # --- main metadata columns ---
    out_df = pd.DataFrame({
        ID_COL: test_df[ID_COL].values,
        "model": model_key,
        "threshold_global": [threshold_global] * n,
        "thresholds_json":  [thresholds_json] * n,
        "pred_labels": [",".join(x) if x else "" for x in pred_labels],
    })

    # --- probabilities ---
    proba_cols = [f"proba_{lab}" for lab in CANON_LABELS]
    out_probs = pd.DataFrame(probs, columns=proba_cols)

    # --- combine (thr cols only exist if vector thresholds) ---
    out_full = pd.concat([out_df, thr_cols_df, out_probs], axis=1)

    path = model_dir / "predictions_test600.csv"
    out_full.to_csv(path, index=False)
    print("✅ Saved:", path.resolve())
    return path, model_dir

In [48]:
def tokenize_dataset(ds, tokenizer, max_length=256, has_labels=True):
    def tok(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

    ds2 = ds.map(tok, batched=True)

    if has_labels:
        # ✅ Force labels to float32 for BCEWithLogitsLoss (multi-label)
        ds2 = ds2.map(lambda b: {"labels": [list(map(float, x)) for x in b["labels"]]}, batched=True)

    cols = ["input_ids", "attention_mask"] + (["labels"] if has_labels else [])
    ds2.set_format(type="torch", columns=cols)
    return ds2

In [49]:
from transformers import TrainingArguments
import inspect

def make_training_args(output_dir, **kwargs):
    """
    Create TrainingArguments in a version-safe way.
    Some versions use evaluation_strategy; others use eval_strategy.
    """
    sig = inspect.signature(TrainingArguments.__init__).parameters

    # rename if needed
    if "evaluation_strategy" in kwargs and "evaluation_strategy" not in sig and "eval_strategy" in sig:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")

    # remove unsupported keys (safety)
    safe_kwargs = {k: v for k, v in kwargs.items() if k in sig}
    dropped = set(kwargs) - set(safe_kwargs)
    if dropped:
        print("⚠️ Dropped unsupported TrainingArguments keys:", dropped)

    return TrainingArguments(output_dir=output_dir, **safe_kwargs)

### BERT

In [64]:
MODEL_NAME_BERT = "bert-base-uncased"
MODEL_KEY_BERT  = "bert-base-uncased"

tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME_BERT, use_fast=True)

ds_tr_bert  = tokenize_dataset(build_ds(X_tr,  Y_tr), tokenizer_bert, max_length=256, has_labels=True)
ds_val_bert = tokenize_dataset(build_ds(X_val, Y_val), tokenizer_bert, max_length=256, has_labels=True)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_BERT,
    num_labels=len(CANON_LABELS),
    problem_type="multi_label_classification"
)

args_bert = make_training_args(
    output_dir=str(OUT_DIR / MODEL_KEY_BERT / "ckpt"),
    evaluation_strategy="epoch",   # will auto-map if needed
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42
)

trainer_bert = Trainer(
    model=bert_model,
    args=args_bert,
    train_dataset=ds_tr_bert,
    eval_dataset=ds_val_bert
)

trainer_bert.train()

val_logits_bert = trainer_bert.predict(ds_val_bert).predictions
best_thr_bert, best_f1_vec_bert = tune_threshold(val_logits_bert, Y_val)

val_probs_bert = sigmoid(val_logits_bert)
Y_val_pred_bert = (val_probs_bert >= best_thr_bert).astype(int)
best_micro_bert = f1_score(Y_val, Y_val_pred_bert, average="micro", zero_division=0)

print("✅ Per-label thresholds tuned. Val micro-F1:", best_micro_bert)
for lbl, thr, f1l in zip(CANON_LABELS, best_thr_bert, best_f1_vec_bert):
    print(f"{lbl:15s} thr={thr:.2f}  val_f1={f1l:.3f}")


Map:   0%|          | 0/3995 [00:00<?, ? examples/s]

Map:   0%|          | 0/3995 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.286000,0.267245
2,0.223900,0.239743
3,0.195200,0.234362


✅ Per-label thresholds tuned. Val micro-F1: 0.8377514570407971
Abuse           thr=0.40  val_f1=0.861
Aggression      thr=0.10  val_f1=0.223
Sexual          thr=0.40  val_f1=0.750
Medical         thr=0.40  val_f1=0.904
Mental Health   thr=0.55  val_f1=0.898
Discrimination  thr=0.15  val_f1=0.438
Pregnancy       thr=0.55  val_f1=0.969
Not Applicable  thr=0.15  val_f1=0.746


In [71]:
X_test = test_df["text"].astype(str).values
ds_test_bert = tokenize_dataset(build_ds(X_test), tokenizer_bert, max_length=256, has_labels=False)

test_logits_bert = trainer_bert.predict(ds_test_bert).predictions
test_probs_bert  = sigmoid(test_logits_bert)

Y_pred_bert      = (test_probs_bert >= best_thr_bert).astype(int)

pred_labels_bert  = mlb.inverse_transform(Y_pred_bert)

pred_path_bert, model_dir_bert = save_predictions(
    MODEL_KEY_BERT,
    best_thr_bert,
    test_probs_bert,
    pred_labels_bert
)

trainer_bert.save_model(str(model_dir_bert / "final_model"))
tokenizer_bert.save_pretrained(str(model_dir_bert / "final_model"))

if np.isscalar(best_thr_bert):
    meta = {
        "model": MODEL_NAME_BERT,
        "threshold_global": float(best_thr_bert),
        "thresholds": None,
        "val_microF1": float(best_micro_bert),
        "labels": CANON_LABELS
    }
else:
    thr_list = [float(x) for x in np.asarray(best_thr_bert).tolist()]
    meta = {
        "model": MODEL_NAME_BERT,
        "threshold_global": None,
        "thresholds": thr_list,  # list aligned with CANON_LABELS
        "thresholds_by_label": {lab: thr for lab, thr in zip(CANON_LABELS, thr_list)},
        "val_microF1": float(best_micro_bert),
        "labels": CANON_LABELS
    }

(model_dir_bert / "meta.json").write_text(json.dumps(meta, indent=2))
print("✅ Wrote meta.json:", (model_dir_bert / "meta.json").resolve())


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

✅ Saved: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/bert-base-uncased/predictions_test600.csv
✅ Wrote meta.json: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/bert-base-uncased/meta.json


#### Bert Metrics

In [21]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

In [22]:
# ====== EDIT THESE ======
PRED_PATH = "/home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/bert-base-uncased/predictions_test600.csv"
GT_PATH   = "/home/ubuntu/TW_MultiLabel_SMP/datasets/test600_labeled_with_ids.csv"

ID_COL = "id"          # change if yours is "id"

PRED_LABEL_COL = "pred_labels"   # change to your predictions column name
GT_LABEL_COL   = "Tags"   # change to your ground truth column name

# Your canonical label set (use the exact same ordering everywhere)
CANON_LABELS = [
    "Abuse", "Aggression", "Sexual", "Medical", "Mental Health",
    "Discrimination", "Pregnancy", "Not Applicable"
]

In [23]:
pred_df = pd.read_csv(PRED_PATH)
gt_df   = pd.read_csv(GT_PATH)

print("pred rows:", len(pred_df), "gt rows:", len(gt_df))

merged = pred_df.merge(gt_df, on=ID_COL, how="inner", suffixes=("_pred", "_gt"))

print("✅ joined rows:", len(merged))
print("pred-only (not matched):", len(pred_df) - merged[ID_COL].nunique())
print("gt-only (not matched):", len(gt_df) - merged[ID_COL].nunique())

display(merged[[ID_COL]].head())

pred rows: 600 gt rows: 600
✅ joined rows: 600
pred-only (not matched): 0
gt-only (not matched): 0


,id
0,kg3jun
1,77d66o
2,1lmsep9
3,1nlkz6v
4,1oa1d0d


In [24]:
def parse_labels(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip()]
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return []
    try:
        val = ast.literal_eval(s)  # handles '["A","B"]'
        if isinstance(val, list):
            return [str(t).strip() for t in val if str(t).strip()]
        return [str(val).strip()]
    except Exception:
        return [t.strip() for t in s.split(",") if t.strip()]

merged["y_pred"] = merged[PRED_LABEL_COL].apply(parse_labels)
merged["y_true"] = merged[GT_LABEL_COL].apply(parse_labels)

display(merged[[ID_COL, "y_true", "y_pred"]].head(10))

,id,y_true,y_pred
0,kg3jun,"[Abuse, Sexual, Mental Health]","[Abuse, Mental Health]"
1,77d66o,"[Abuse, Sexual, Mental Health]","[Abuse, Sexual, Mental Health]"
2,1lmsep9,[Not Applicable],[Not Applicable]
3,1nlkz6v,"[Medical, Mental Health, Pregnancy]","[Medical, Mental Health, Pregnancy]"
4,1oa1d0d,"[Medical, Mental Health, Pregnancy]","[Medical, Mental Health, Pregnancy]"
5,9dahpm,"[Abuse, Sexual, Mental Health]","[Abuse, Sexual, Mental Health]"
6,ola3vt,"[Abuse, Sexual, Mental Health]","[Abuse, Sexual, Mental Health]"
7,1o5thk7,"[Pregnancy, Medical, Mental Health]","[Medical, Mental Health, Pregnancy]"
8,n579qx,"[Abuse, Sexual]","[Abuse, Mental Health]"
9,ln5spt,"[Abuse, Sexual, Mental Health]","[Abuse, Sexual, Mental Health]"


In [25]:
mlb = MultiLabelBinarizer(classes=CANON_LABELS)

Y_true = mlb.fit_transform(merged["y_true"])
Y_pred = mlb.transform(merged["y_pred"])

print("Y_true shape:", Y_true.shape, "Y_pred shape:", Y_pred.shape)

Y_true shape: (600, 8) Y_pred shape: (600, 8)


/home/ubuntu/TW_MultiLabel_SMP/tw_env/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1007: UserWarning: unknown class(es) ['$325. after', 'an antibiotic', 'and a 800 mg ibuprofen for pain. i had to wait 30 minutes for the medicine to kick in', 'and checked my vitals again. i stayed for 10 minutes and then they let me go home. once i was reunited with my boyfriend', 'and felt really weak. the actual procedure lasted only 5 minutes', 'and i could not eat or drink anything 8 hours prior to my appointment. i returned back to planned parenthood where they collected the rest of the payment', 'and i have an intense phobia of getting sick (vomit). i also chose not to have sedation', 'and i was on my way back home. from the first appointment to the actual procedure date', 'and it is now 10 pm. when we got home', 'and overall sadness. i was in pain and i felt as if the doctor was poking and prodding around in me which felt a little too invasive for me. but', 'and the doctor and nur

In [26]:
# Exact match accuracy = all labels exactly correct for a row
exact_match = accuracy_score(Y_true, Y_pred)

# Micro / Macro P/R/F1
p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
    Y_true, Y_pred, average="micro", zero_division=0
)
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    Y_true, Y_pred, average="macro", zero_division=0
)

print(f"✅ Exact match accuracy: {exact_match:.4f}")
print(f"✅ Micro  P/R/F1: {p_micro:.4f} / {r_micro:.4f} / {f1_micro:.4f}")
print(f"✅ Macro  P/R/F1: {p_macro:.4f} / {r_macro:.4f} / {f1_macro:.4f}")

✅ Exact match accuracy: 0.5500
✅ Micro  P/R/F1: 0.8531 / 0.8548 / 0.8539
✅ Macro  P/R/F1: 0.7231 / 0.6466 / 0.6552


In [27]:
p, r, f1, support = precision_recall_fscore_support(
    Y_true, Y_pred, average=None, zero_division=0
)

per_label = pd.DataFrame({
    "label": CANON_LABELS,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support": support
}).sort_values("f1", ascending=False)

display(per_label)

,label,precision,recall,f1,support
6,Pregnancy,0.932722,0.987055,0.959119,309
3,Medical,0.900398,0.904000,0.902196,250
0,Abuse,0.837321,0.921053,0.877193,190
4,Mental Health,0.778004,0.982005,0.868182,389
2,Sexual,0.840708,0.678571,0.750988,140
7,Not Applicable,0.924051,0.618644,0.741117,118
5,Discrimination,0.571429,0.081633,0.142857,49
1,Aggression,0.000000,0.000000,0.000000,29


In [29]:
print("\n" + "="*60)
print("BERT (bert-base-uncased) — Classification Report")
print("="*60)

print(classification_report(
    Y_true, Y_pred,
    target_names=CANON_LABELS,
    zero_division=0
))


BERT (bert-base-uncased) — Classification Report
                precision    recall  f1-score   support

         Abuse       0.84      0.92      0.88       190
    Aggression       0.00      0.00      0.00        29
        Sexual       0.84      0.68      0.75       140
       Medical       0.90      0.90      0.90       250
 Mental Health       0.78      0.98      0.87       389
Discrimination       0.57      0.08      0.14        49
     Pregnancy       0.93      0.99      0.96       309
Not Applicable       0.92      0.62      0.74       118

     micro avg       0.85      0.85      0.85      1474
     macro avg       0.72      0.65      0.66      1474
  weighted avg       0.83      0.85      0.83      1474
   samples avg       0.84      0.84      0.83      1474



### RoBERTa

In [73]:
MODEL_NAME_ROB = "roberta-base"
MODEL_KEY_ROB  = "roberta-base"

tokenizer_rob = AutoTokenizer.from_pretrained(MODEL_NAME_ROB, use_fast=True)

ds_tr_rob  = tokenize_dataset(build_ds(X_tr,  Y_tr), tokenizer_rob, max_length=256, has_labels=True)
ds_val_rob = tokenize_dataset(build_ds(X_val, Y_val), tokenizer_rob, max_length=256, has_labels=True)

rob_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_ROB,
    num_labels=len(CANON_LABELS),
    problem_type="multi_label_classification"
)

args_rob = TrainingArguments(
    output_dir=str(OUT_DIR / MODEL_KEY_ROB / "ckpt"),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42
)

trainer_rob = Trainer(model=rob_model, args=args_rob, train_dataset=ds_tr_rob, eval_dataset=ds_val_rob)
trainer_rob.train()

val_logits_rob = trainer_rob.predict(ds_val_rob).predictions
best_thr_rob, best_f1_vec_rob = tune_threshold(val_logits_rob, Y_val)

val_probs_rob = sigmoid(val_logits_rob)
Y_val_pred_rob = (val_probs_rob >= best_thr_rob).astype(int)
best_micro_rob = f1_score(Y_val, Y_val_pred_rob, average="micro", zero_division=0)

print("✅ Per-label thresholds tuned. Val micro-F1:", best_micro_rob)
for lbl, thr, f1l in zip(CANON_LABELS, best_thr_rob, best_f1_vec_rob):
    print(f"{lbl:15s} thr={thr:.2f}  val_f1={f1l:.3f}")

Map:   0%|          | 0/3995 [00:00<?, ? examples/s]

Map:   0%|          | 0/3995 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.260800,0.256007
2,0.214900,0.232665
3,0.182700,0.224617


✅ Per-label thresholds tuned. Val micro-F1: 0.860456126787785
Abuse           thr=0.35  val_f1=0.870
Aggression      thr=0.15  val_f1=0.262
Sexual          thr=0.35  val_f1=0.747
Medical         thr=0.25  val_f1=0.909
Mental Health   thr=0.55  val_f1=0.909
Discrimination  thr=0.20  val_f1=0.436
Pregnancy       thr=0.75  val_f1=0.969
Not Applicable  thr=0.20  val_f1=0.773


In [74]:
ds_test_rob = tokenize_dataset(build_ds(X_test), tokenizer_rob, max_length=256, has_labels=False)

test_logits_rob = trainer_rob.predict(ds_test_rob).predictions
test_probs_rob  = sigmoid(test_logits_rob)

Y_pred_rob      = (test_probs_rob >= best_thr_rob).astype(int)
pred_labels_rob  = mlb.inverse_transform(Y_pred_rob)

pred_path_rob, model_dir_rob = save_predictions(
    MODEL_KEY_ROB,
    best_thr_rob,
    test_probs_rob,
    pred_labels_rob
)

trainer_rob.save_model(str(model_dir_rob / "final_model"))
tokenizer_rob.save_pretrained(str(model_dir_rob / "final_model"))

if np.isscalar(best_thr_rob):
    meta = {
        "model": MODEL_NAME_ROB,
        "threshold_global": float(best_thr_rob),
        "thresholds": None,
        "val_microF1": float(best_micro_rob),
        "labels": CANON_LABELS
    }
else:
    thr_list = [float(x) for x in np.asarray(best_thr_rob).tolist()]
    meta = {
        "model": MODEL_NAME_ROB,
        "threshold_global": None,
        "thresholds": thr_list,  
        "thresholds_by_label": {lab: thr for lab, thr in zip(CANON_LABELS, thr_list)},
        "val_microF1": float(best_micro_rob),
        "labels": CANON_LABELS
    }

(model_dir_rob / "meta.json").write_text(json.dumps(meta, indent=2))
print("✅ Wrote meta.json:", (model_dir_rob / "meta.json").resolve())

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

✅ Saved: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/roberta-base/predictions_test600.csv
✅ Wrote meta.json: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/roberta-base/meta.json


### Metrics

In [85]:
# ====== EDIT THESE ======
PRED_PATH = "/home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/roberta-base/predictions_test600.csv"
GT_PATH   = "/home/ubuntu/TW_MultiLabel_SMP/datasets/test600_labeled_with_ids.csv"


# ===== Model prediction files =====
MODEL_FILES = {
    "TF-IDF + SVM": "/home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/predictions_tfidf_svm_test600.csv",
    "TF-IDF + LR": "/home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/predictions_tfidf_lr_test600.csv",
    "BERT": "/home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/bert-base-uncased/predictions_test600.csv",
    "RoBERTa":   "/home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/roberta-base/predictions_test600.csv",
    # "DistilBERT": "/home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/distilbert-base-uncased/predictions_test600.csv",
    "XLNet":     "/home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/xlnet-base-cased/predictions_test600.csv",
}

ID_COL = "id"          # change if yours is "id"

PRED_LABEL_COL = "pred_labels"   # change to your predictions column name
GT_LABEL_COL   = "Tags"   # change to your ground truth column name

# Your canonical label set (use the exact same ordering everywhere)
CANON_LABELS = [
    "Abuse", "Aggression", "Sexual", "Medical", "Mental Health",
    "Discrimination", "Pregnancy", "Not Applicable"
]

In [83]:
import ast
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# --- OPTIONAL but recommended: normalize label strings to your canonical list ---
# Build a mapping like {"mental health": "Mental Health", ...}
CANON_MAP = {c.lower().strip(): c for c in CANON_LABELS}

def normalize_label(lbl: str):
    s = str(lbl).strip()
    if not s:
        return None
    key = s.lower().strip()
    return CANON_MAP.get(key, s)  # keep original if not found

def parse_labels(x):
    """
    Handles:
    - NaN
    - python list already
    - stringified list: "['A','B']"
    - comma-separated: "A, B"
    Returns: list[str]
    """
    if pd.isna(x):
        return []

    # already list-like
    if isinstance(x, list):
        out = []
        for t in x:
            nt = normalize_label(t)
            if nt:
                out.append(nt)
        return out

    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return []

    # try parsing as python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            out = []
            for t in val:
                nt = normalize_label(t)
                if nt:
                    out.append(nt)
            return out
        else:
            nt = normalize_label(val)
            return [nt] if nt else []
    except Exception:
        # fallback: comma-separated
        out = []
        for t in s.split(","):
            nt = normalize_label(t)
            if nt:
                out.append(nt)
        return out


def evaluate_model(name, pred_path, gt_df):
    pred_df = pd.read_csv(pred_path)

    # Merge predictions and ground truth on ID
    merged = pred_df.merge(gt_df, on=ID_COL, how="inner")

    # Parse labels
    merged["y_true"] = merged[GT_LABEL_COL].apply(parse_labels)
    merged["y_pred"] = merged[PRED_LABEL_COL].apply(parse_labels)

    # Binarize using a FIXED class order (your CANON_LABELS)
    mlb = MultiLabelBinarizer(classes=CANON_LABELS)
    Y_true = mlb.fit_transform(merged["y_true"])
    Y_pred = mlb.transform(merged["y_pred"])

    # Metrics
    exact_match = accuracy_score(Y_true, Y_pred)

    micro_p = precision_score(Y_true, Y_pred, average="micro", zero_division=0)
    micro_r = recall_score(Y_true, Y_pred, average="micro", zero_division=0)
    micro_f1 = f1_score(Y_true, Y_pred, average="micro", zero_division=0)

    macro_p = precision_score(Y_true, Y_pred, average="macro", zero_division=0)
    macro_r = recall_score(Y_true, Y_pred, average="macro", zero_division=0)
    macro_f1 = f1_score(Y_true, Y_pred, average="macro", zero_division=0)

    weighted_p = precision_score(Y_true, Y_pred, average="weighted", zero_division=0)
    weighted_r = recall_score(Y_true, Y_pred, average="weighted", zero_division=0)
    weighted_f1 = f1_score(Y_true, Y_pred, average="weighted", zero_division=0)

    # Print with heading
    print("\n" + "="*70)
    print(f"{name} — Classification Report")
    print("="*70)
    print(classification_report(
        Y_true, Y_pred,
        target_names=CANON_LABELS,
        zero_division=0
    ))

    # Return a full row for your summary table
    return {
        "model": name,

        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "micro_f1": micro_f1,

        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,

        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
        "weighted_f1": weighted_f1,
    }

In [81]:
gt_df = pd.read_csv(GT_PATH)
print("GT rows:", len(gt_df))

GT rows: 600


In [86]:
results = []

for model_name, path in MODEL_FILES.items():
    res = evaluate_model(model_name, path, gt_df)
    results.append(res)

summary_df = pd.DataFrame(results)
display(summary_df)

/home/ubuntu/TW_MultiLabel_SMP/tw_env/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1007: UserWarning: unknown class(es) ['$325. after', 'an antibiotic', 'and a 800 mg ibuprofen for pain. i had to wait 30 minutes for the medicine to kick in', 'and checked my vitals again. i stayed for 10 minutes and then they let me go home. once i was reunited with my boyfriend', 'and felt really weak. the actual procedure lasted only 5 minutes', 'and i could not eat or drink anything 8 hours prior to my appointment. i returned back to planned parenthood where they collected the rest of the payment', 'and i have an intense phobia of getting sick (vomit). i also chose not to have sedation', 'and i was on my way back home. from the first appointment to the actual procedure date', 'and it is now 10 pm. when we got home', 'and overall sadness. i was in pain and i felt as if the doctor was poking and prodding around in me which felt a little too invasive for me. but', 'and the doctor and nur


TF-IDF + SVM — Classification Report
                precision    recall  f1-score   support

         Abuse       0.82      0.89      0.86       190
    Aggression       1.00      0.07      0.13        29
        Sexual       0.83      0.75      0.79       140
       Medical       0.90      0.86      0.88       250
 Mental Health       0.78      0.95      0.86       389
Discrimination       0.43      0.18      0.26        49
     Pregnancy       0.93      0.96      0.95       309
Not Applicable       0.91      0.50      0.64       118

     micro avg       0.85      0.83      0.84      1474
     macro avg       0.83      0.65      0.67      1474
  weighted avg       0.85      0.83      0.82      1474
   samples avg       0.82      0.80      0.79      1474


TF-IDF + LR — Classification Report
                precision    recall  f1-score   support

         Abuse       0.76      0.89      0.82       190
    Aggression       0.11      1.00      0.20        29
        Sexual       0.74

/home/ubuntu/TW_MultiLabel_SMP/tw_env/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1007: UserWarning: unknown class(es) ['$325. after', 'an antibiotic', 'and a 800 mg ibuprofen for pain. i had to wait 30 minutes for the medicine to kick in', 'and checked my vitals again. i stayed for 10 minutes and then they let me go home. once i was reunited with my boyfriend', 'and felt really weak. the actual procedure lasted only 5 minutes', 'and i could not eat or drink anything 8 hours prior to my appointment. i returned back to planned parenthood where they collected the rest of the payment', 'and i have an intense phobia of getting sick (vomit). i also chose not to have sedation', 'and i was on my way back home. from the first appointment to the actual procedure date', 'and it is now 10 pm. when we got home', 'and overall sadness. i was in pain and i felt as if the doctor was poking and prodding around in me which felt a little too invasive for me. but', 'and the doctor and nur

,model,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
0,TF-IDF + SVM,0.845148,0.833107,0.839084,0.825644,0.646795,0.670463,0.845957,0.833107,0.822511
1,TF-IDF + LR,0.710306,0.864993,0.780055,0.665396,0.794655,0.682326,0.797836,0.864993,0.821574
2,BERT,0.761338,0.911126,0.829524,0.684027,0.836094,0.729254,0.812306,0.911126,0.853184
3,RoBERTa,0.795455,0.902307,0.845518,0.706262,0.802885,0.740145,0.820572,0.902307,0.855176
4,XLNet,0.836294,0.894166,0.864262,0.751513,0.778850,0.762733,0.839364,0.894166,0.863793


### DistilBERT

In [75]:
MODEL_NAME_DISTIL = "distilbert-base-uncased"
MODEL_KEY_DISTIL  = "distilbert-base-uncased"

tokenizer_distil = AutoTokenizer.from_pretrained(MODEL_NAME_DISTIL, use_fast=True)

ds_tr_distil  = tokenize_dataset(build_ds(X_tr,  Y_tr), tokenizer_distil, max_length=256, has_labels=True)
ds_val_distil = tokenize_dataset(build_ds(X_val, Y_val), tokenizer_distil, max_length=256, has_labels=True)

distil_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_DISTIL,
    num_labels=len(CANON_LABELS),
    problem_type="multi_label_classification"
)

args_distil = TrainingArguments(
    output_dir=str(OUT_DIR / MODEL_KEY_DISTIL / "ckpt"),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    num_train_epochs=3,
    per_device_train_batch_size=16,   # DistilBERT is lighter
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42
)

trainer_distil = Trainer(model=distil_model, args=args_distil, train_dataset=ds_tr_distil, eval_dataset=ds_val_distil)
trainer_distil.train()

val_logits_distil = trainer_distil.predict(ds_val_distil).predictions

best_thr_distil, best_f1_vec_distil = tune_threshold(val_logits_distil, Y_val)

val_probs_distil = sigmoid(val_logits_distil)
Y_val_pred_distil = (val_probs_distil >= best_thr_distil).astype(int)
best_micro_distil = f1_score(Y_val, Y_val_pred_distil, average="micro", zero_division=0)

print(f"✅ DistilBERT val_microF1={best_micro_distil:.4f}")
for lbl, thr, f1l in zip(CANON_LABELS, best_thr_distil, best_f1_vec_distil):
    print(f"{lbl:15s} thr={thr:.2f}  val_f1={f1l:.3f}")

Map:   0%|          | 0/3995 [00:00<?, ? examples/s]

Map:   0%|          | 0/3995 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.298400,0.264956
2,0.238000,0.237326
3,0.193600,0.232874


✅ DistilBERT val_microF1=0.8336
Abuse           thr=0.40  val_f1=0.856
Aggression      thr=0.10  val_f1=0.229
Sexual          thr=0.45  val_f1=0.713
Medical         thr=0.45  val_f1=0.900
Mental Health   thr=0.50  val_f1=0.892
Discrimination  thr=0.15  val_f1=0.359
Pregnancy       thr=0.65  val_f1=0.970
Not Applicable  thr=0.20  val_f1=0.759


In [76]:
ds_test_distil = tokenize_dataset(build_ds(X_test), tokenizer_distil, max_length=256, has_labels=False)

test_logits_distil = trainer_distil.predict(ds_test_distil).predictions
test_probs_distil  = sigmoid(test_logits_distil)

Y_pred_distil      = (test_probs_distil >= best_thr_distil).astype(int)
pred_labels_distil  = mlb.inverse_transform(Y_pred_distil)

pred_path_distil, model_dir_distil = save_predictions(
    MODEL_KEY_DISTIL,
    best_thr_distil,
    test_probs_distil,
    pred_labels_distil
)

trainer_distil.save_model(str(model_dir_distil / "final_model"))
tokenizer_distil.save_pretrained(str(model_dir_distil / "final_model"))

if np.isscalar(best_thr_distil):
    meta = {
        "model": MODEL_NAME_DISTIL,
        "threshold_global": float(best_thr_distil),
        "thresholds": None,
        "val_microF1": float(best_micro_distil),
        "labels": CANON_LABELS
    }
else:
    thr_list = [float(x) for x in np.asarray(best_thr_distil).tolist()]
    meta = {
        "model": MODEL_NAME_DISTIL,
        "threshold_global": None,
        "thresholds": thr_list,
        "thresholds_by_label": {lab: thr for lab, thr in zip(CANON_LABELS, thr_list)},
        "val_microF1": float(best_micro_distil),
        "labels": CANON_LABELS
    }

(model_dir_distil / "meta.json").write_text(json.dumps(meta, indent=2))
print("✅ Wrote meta.json:", (model_dir_distil / "meta.json").resolve())


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

✅ Saved: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/distilbert-base-uncased/predictions_test600.csv
✅ Wrote meta.json: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/distilbert-base-uncased/meta.json


### XLNet

In [77]:
MODEL_NAME_XLNET = "xlnet-base-cased"
MODEL_KEY_XLNET  = "xlnet-base-cased"

tokenizer_xlnet = AutoTokenizer.from_pretrained(MODEL_NAME_XLNET, use_fast=True)

# XLNet can be heavier; drop max_length if OOM
ds_tr_xlnet  = tokenize_dataset(build_ds(X_tr,  Y_tr), tokenizer_xlnet, max_length=256, has_labels=True)
ds_val_xlnet = tokenize_dataset(build_ds(X_val, Y_val), tokenizer_xlnet, max_length=256, has_labels=True)

xlnet_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_XLNET,
    num_labels=len(CANON_LABELS),
    problem_type="multi_label_classification"
)

args_xlnet = TrainingArguments(
    output_dir=str(OUT_DIR / MODEL_KEY_XLNET / "ckpt"),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42
)

trainer_xlnet = Trainer(model=xlnet_model, args=args_xlnet, train_dataset=ds_tr_xlnet, eval_dataset=ds_val_xlnet)
trainer_xlnet.train()

val_logits_xlnet = trainer_xlnet.predict(ds_val_xlnet).predictions

best_thr_xlnet, best_f1_vec_xlnet = tune_threshold(val_logits_xlnet, Y_val)

val_probs_xlnet = sigmoid(val_logits_xlnet)
Y_val_pred_xlnet = (val_probs_xlnet >= best_thr_xlnet).astype(int)
best_micro_xlnet = f1_score(Y_val, Y_val_pred_xlnet, average="micro", zero_division=0)

print(f"✅ XLNet val_microF1={best_micro_xlnet:.4f}")
for lbl, thr, f1l in zip(CANON_LABELS, best_thr_xlnet, best_f1_vec_xlnet):
    print(f"{lbl:15s} thr={thr:.2f}  val_f1={f1l:.3f}")

Map:   0%|          | 0/3995 [00:00<?, ? examples/s]

Map:   0%|          | 0/3995 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Some weights of XLNetForSequenceClassification were not initialized from the model checkpoint at xlnet-base-cased and are newly initialized: ['logits_proj.bias', 'logits_proj.weight', 'sequence_summary.summary.bias', 'sequence_summary.summary.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.260100,0.241280
2,0.207500,0.223583
3,0.171400,0.223375


✅ XLNet val_microF1=0.8685
Abuse           thr=0.70  val_f1=0.876
Aggression      thr=0.15  val_f1=0.280
Sexual          thr=0.45  val_f1=0.747
Medical         thr=0.30  val_f1=0.901
Mental Health   thr=0.60  val_f1=0.910
Discrimination  thr=0.25  val_f1=0.473
Pregnancy       thr=0.50  val_f1=0.970
Not Applicable  thr=0.20  val_f1=0.766


In [78]:
ds_test_xlnet = tokenize_dataset(build_ds(X_test), tokenizer_xlnet, max_length=256, has_labels=False)

test_logits_xlnet = trainer_xlnet.predict(ds_test_xlnet).predictions
test_probs_xlnet  = sigmoid(test_logits_xlnet)

Y_pred_xlnet      = (test_probs_xlnet >= best_thr_xlnet).astype(int)
pred_labels_xlnet  = mlb.inverse_transform(Y_pred_xlnet)

pred_path_xlnet, model_dir_xlnet = save_predictions(
    MODEL_KEY_XLNET,
    best_thr_xlnet,
    test_probs_xlnet,
    pred_labels_xlnet
)

trainer_xlnet.save_model(str(model_dir_xlnet / "final_model"))
tokenizer_xlnet.save_pretrained(str(model_dir_xlnet / "final_model"))

if np.isscalar(best_thr_xlnet):
    meta = {
        "model": MODEL_NAME_XLNET,
        "threshold_global": float(best_thr_xlnet),
        "thresholds": None,
        "val_microF1": float(best_micro_xlnet),
        "labels": CANON_LABELS
    }
else:
    thr_list = [float(x) for x in np.asarray(best_thr_xlnet).tolist()]
    meta = {
        "model": MODEL_NAME_XLNET,
        "threshold_global": None,
        "thresholds": thr_list,
        "thresholds_by_label": {lab: thr for lab, thr in zip(CANON_LABELS, thr_list)},
        "val_microF1": float(best_micro_xlnet),
        "labels": CANON_LABELS
    }

(model_dir_xlnet / "meta.json").write_text(json.dumps(meta, indent=2))
print("✅ Wrote meta.json:", (model_dir_xlnet / "meta.json").resolve())


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

✅ Saved: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/xlnet-base-cased/predictions_test600.csv
✅ Wrote meta.json: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/xlnet-base-cased/meta.json


### Transformer Predictions

In [24]:
paths = [pred_path_bert, pred_path_rob, pred_path_distil, pred_path_xlnet]

merged = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
MERGED_PATH = OUT_DIR / "all_transformer_predictions_test600.csv"
merged.to_csv(MERGED_PATH, index=False)

print("✅ Saved merged predictions:", MERGED_PATH.resolve())
display(merged.head(5))

✅ Saved merged predictions: /home/ubuntu/TW_MultiLabel_SMP/model_outputs/traditional_ml_inference/all_transformer_predictions_test600.csv


,id,model,threshold,pred_labels,proba_Abuse,proba_Aggression,proba_Sexual,proba_Medical,proba_Mental Health,proba_Discrimination,proba_Pregnancy,proba_Not Applicable
0,kg3jun,bert-base-uncased,0.5,"Abuse,Mental Health",0.972046,0.077378,0.281801,0.024704,0.954478,0.162646,0.042642,0.021045
1,77d66o,bert-base-uncased,0.5,"Abuse,Sexual,Mental Health",0.975249,0.155591,0.882428,0.062560,0.959231,0.117775,0.056862,0.022244
2,1lmsep9,bert-base-uncased,0.5,Not Applicable,0.071332,0.029312,0.025083,0.027585,0.066085,0.047603,0.033463,0.901401
3,1nlkz6v,bert-base-uncased,0.5,"Medical,Mental Health,Pregnancy",0.013637,0.007786,0.012723,0.906653,0.848596,0.056966,0.987080,0.020139
4,1oa1d0d,bert-base-uncased,0.5,"Medical,Mental Health,Pregnancy",0.016153,0.016850,0.031619,0.985440,0.754553,0.068289,0.981979,0.023599


## LLMS

### GPT